# 🧠 Llamar a un LLM desde Python (triage de alertas)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/florvela/IA-y-automatizacion-en-seguridad-defensiva/blob/main/codigos-de-ejemplo/clase_en_vivo/live_06_llm.ipynb)

**Presentar en la diapositiva 152** ("Python para LLMs").

Vemos el código mínimo para llamar a un LLM y un caso de uso real: **triage de una alerta**.
La respuesta siempre se **verifica** — el LLM puede alucinar.

<img src="https://raw.githubusercontent.com/florvela/IA-y-automatizacion-en-seguridad-defensiva/main/08-nlp-y-llms/images/mas-probable.png" width="460"/>


## 1. Setup
La API key **nunca** va en el código. La pedimos de forma segura.

In [ ]:
!pip install anthropic -q

In [ ]:
import os, getpass
os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Pegá tu API key de Anthropic: ")

## 2. Llamada mínima a la API

In [ ]:
import anthropic

client = anthropic.Anthropic()   # lee ANTHROPIC_API_KEY del entorno

resp = client.messages.create(
    model="claude-opus-5",
    max_tokens=200,
    messages=[{"role": "user", "content": "¿Qué es un IOC en seguridad? Respondé en 2 frases."}],
)
print(resp.content[0].text)

## 3. Caso de uso: triage de una alerta
Le pedimos categorías **cerradas** y salida en **JSON** → así alucina menos y podemos parsearlo.

In [ ]:
import json

alerta = "Multiples intentos de login fallidos desde 185.220.101.47 hacia la cuenta admin (Tor exit node)"

prompt = f"""Sos un analista SOC. Clasificá esta alerta y respondé SOLO con JSON:
{{"tipo": "FUERZA_BRUTA|EXFILTRACION|MALWARE|OTRO",
  "severidad": "CRITICA|ALTA|MEDIA|BAJA",
  "falso_positivo_probable": true/false,
  "accion_recomendada": "..."}}

Alerta: {alerta}"""

resp = client.messages.create(
    model="claude-opus-5",
    max_tokens=300,
    messages=[{"role": "user", "content": prompt}],
)

datos = json.loads(resp.content[0].text)
print(json.dumps(datos, indent=2, ensure_ascii=False))

## ⚠️ Siempre verificar
El LLM pre-clasifica y ahorra tiempo, **pero no reemplaza al analista**.
Los IOCs (IPs, hashes) que devuelva un LLM **siempre** se verifican contra fuentes reales
(VirusTotal, el SIEM, la CMDB) antes de usarlos en una regla o un bloqueo.